# Post Delta Analysis

Run post-processing after `run_delta_debug.py` experiment: sequence analysis, circuit verification, and TVD visualization.


In [ ]:
from pathlib import Path
import shutil
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from quantum.analyze_problematic_segments import analyze_and_generate_circuits
from quantum import circuit_verification
from quantum.tvd_visualization import load_results, visualize_tvd_with_sequences, print_detailed_summary
import pandas as pd


In [ ]:
# Configure target algorithm directory
algorithm_dir = REPO_ROOT / 'artifacts' / 'grover3-3'
tmp_dir = REPO_ROOT / 'tmp'
tmp_dir.mkdir(parents=True, exist_ok=True)
algorithm_dir


In [ ]:
# Step A: analyze all delta reports and generate candidate circuits
analysis = analyze_and_generate_circuits(str(algorithm_dir), str(tmp_dir))
analysis is not None


In [ ]:
# Step B: verify generated circuits on noisy simulator vs real device
circuit_verification.verify_circuits(
    config_file=str(REPO_ROOT / 'quantum_config.json'),
    circuits_dir=str(tmp_dir / 'circuits'),
    output_file=str(tmp_dir / 'circuit_comparison_results.json'),
)


In [ ]:
# Step C: visualize TVD and save outputs to artifacts directory
comparison_results, sequence_analysis = load_results(str(tmp_dir))
df = visualize_tvd_with_sequences(
    comparison_results,
    sequence_analysis,
    output_dir=str(algorithm_dir),
    show_plot=True,
)
print_detailed_summary(df)
df.to_csv(algorithm_dir / 'tvd_summary.csv', index=False)
print('Saved:', algorithm_dir / 'tvd_summary.csv')


In [ ]:
# Optional: copy tmp outputs back into algorithm dir
for name in ['sequence_analysis.json', 'circuit_comparison_results.json']:
    src = tmp_dir / name
    if src.exists():
        dst = algorithm_dir / name
        shutil.copy2(src, dst)
        print('Copied:', dst)
